In [4]:
import pandas as pd
from sklearn.linear_model import Ridge
import numpy as np
from pathlib import Path
from aieng.forecasting.data import DataService, SeriesMetadata
from aieng.forecasting.data.adapters import StatCanAdapter
import pandas as pd
from sklearn.linear_model import Ridge
import numpy as np

ROOT = Path.cwd().resolve().parents[1]

# Initialize DataService
svc = DataService()

# Register gasoline
CPI_TABLE_ID = "18-10-0004-11"
CACHE_DIR = ROOT / "data" / "statcan"

gasoline_adapter = StatCanAdapter(
    table_id=CPI_TABLE_ID,
    member_filter={"GEO": "Canada", "Products and product groups": "Gasoline"},
    cache_dir=CACHE_DIR,
)

svc.register(
    "cpi_gasoline_canada",
    gasoline_adapter,
    SeriesMetadata(
        series_id="cpi_gasoline_canada",
        description="CPI Gasoline, Canada (2002=100)",
        source="StatCan table 18-10-0004-11",
        units="Index 2002=100",
        frequency="MS",
        table_id=CPI_TABLE_ID,
    ),
)

# Now try to get gasoline
from datetime import datetime, timezone

gasoline = svc.get_series(
    'cpi_gasoline_canada',
    as_of=datetime.now(tz=timezone.utc).replace(tzinfo=None)
)

print(f"Gasoline shape: {gasoline.shape}")
print(gasoline.head())


# Get gasoline (you already have this)
#gasoline = svc.get_series('cpi_gasoline_canada')
print(f"Gasoline shape: {gasoline.shape}")
print(gasoline.head())

# Try to get Industrial Production from FRED
try:
    ind_prod = svc.get_series('INDPRO')
    print(f"Industrial Production shape: {ind_prod.shape}")
except Exception as e:
    print(f"Error: {e}")

Gasoline shape: (929, 3)
   timestamp  value released_at
0 1949-01-01   11.7  1949-01-22
1 1949-02-01   11.7  1949-02-22
2 1949-03-01   11.7  1949-03-22
3 1949-04-01   11.8  1949-04-22
4 1949-05-01   11.8  1949-05-22
Gasoline shape: (929, 3)
   timestamp  value released_at
0 1949-01-01   11.7  1949-01-22
1 1949-02-01   11.7  1949-02-22
2 1949-03-01   11.7  1949-03-22
3 1949-04-01   11.8  1949-04-22
4 1949-05-01   11.8  1949-05-22
Error: DataService.get_series() missing 1 required positional argument: 'as_of'


In [5]:
# Look at what data adapters exist
from aieng.forecasting.data import adapters
print(dir(adapters))

['BaseAdapter', 'FREDAdapter', 'StatCanAdapter', 'YFinanceDailyAdapter', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'base', 'fred', 'statcan', 'yfinance']


In [6]:
from aieng.forecasting.data.adapters import FREDAdapter

# Register Industrial Production from FRED
FRED_CACHE_DIR = ROOT / "data" / "fred"

indpro_adapter = FREDAdapter(
    series_id="INDPRO",  # US Industrial Production
    cache_dir=FRED_CACHE_DIR,
)

svc.register(
    "indpro",
    indpro_adapter,
    SeriesMetadata(
        series_id="indpro",
        description="US Industrial Production Index",
        source="FRED INDPRO",
        units="Index 2017=100",
        frequency="MS",
    ),
)

# Try to fetch it
indpro = svc.get_series(
    'indpro',
    as_of=datetime.now(tz=timezone.utc).replace(tzinfo=None)
)
print(f"Industrial Production shape: {indpro.shape}")
print(indpro.head())
print(indpro.tail())

Industrial Production shape: (1289, 3)
   timestamp   value released_at
0 1919-01-01  4.8739  1919-01-01
1 1919-02-01  4.6585  1919-02-01
2 1919-03-01  4.5238  1919-03-01
3 1919-04-01  4.6046  1919-04-01
4 1919-05-01  4.6315  1919-05-01
      timestamp     value released_at
1284 2026-01-01  101.1235  2026-01-01
1285 2026-02-01  101.9493  2026-02-01
1286 2026-03-01  101.6273  2026-03-01
1287 2026-04-01  102.5090  2026-04-01
1288 2026-05-01  102.6475  2026-05-01


In [7]:
# Merge gasoline and industrial production on timestamp
merged = pd.merge(
    gasoline[['timestamp', 'value']].rename(columns={'value': 'gasoline'}),
    indpro[['timestamp', 'value']].rename(columns={'value': 'indpro'}),
    on='timestamp',
    how='inner'  # only keep dates where BOTH exist
)

print(f"Merged shape: {merged.shape}")
print(merged.head())
print(merged.tail())

Merged shape: (929, 3)
   timestamp  gasoline   indpro
0 1949-01-01      11.7  14.1639
1 1949-02-01      11.7  14.0292
2 1949-03-01      11.7  13.7600
3 1949-04-01      11.8  13.6792
4 1949-05-01      11.8  13.4907
     timestamp  gasoline    indpro
924 2026-01-01     189.5  101.1235
925 2026-02-01     196.3  101.9493
926 2026-03-01     238.0  101.6273
927 2026-04-01     259.3  102.5090
928 2026-05-01     273.7  102.6475


In [8]:
# Work with month-over-month changes, not levels
merged = merged.sort_values('timestamp').reset_index(drop=True)

# Percent change in gasoline (what we want to predict)
merged['gasoline_pct'] = merged['gasoline'].pct_change()

# Percent change in industrial production (our predictor)
merged['indpro_pct'] = merged['indpro'].pct_change()

# Drop the first row (NaN from pct_change)
model_data = merged.dropna().reset_index(drop=True)

print(f"Model data shape: {model_data.shape}")
print(model_data[['timestamp', 'gasoline_pct', 'indpro_pct']].head())
print()
print("Correlation between gasoline change and indpro change:")
print(model_data[['gasoline_pct', 'indpro_pct']].corr())

Model data shape: (928, 5)
   timestamp  gasoline_pct  indpro_pct
0 1949-02-01      0.000000   -0.009510
1 1949-03-01      0.000000   -0.019189
2 1949-04-01      0.008547   -0.005872
3 1949-05-01      0.000000   -0.013780
4 1949-06-01      0.000000   -0.001994

Correlation between gasoline change and indpro change:
              gasoline_pct  indpro_pct
gasoline_pct      1.000000    0.134033
indpro_pct        0.134033    1.000000


In [9]:
# Register WTI crude oil from FRED
crude_adapter = FREDAdapter(
    series_id="DCOILWTICO",
    cache_dir=FRED_CACHE_DIR,
)

svc.register(
    "crude",
    crude_adapter,
    SeriesMetadata(
        series_id="crude",
        description="WTI Crude Oil Price",
        source="FRED DCOILWTICO",
        units="USD/barrel",
        frequency="MS",
    ),
)

# Fetch it
crude = svc.get_series(
    'crude',
    as_of=datetime.now(tz=timezone.utc).replace(tzinfo=None)
)
print(f"Crude shape: {crude.shape}")
print(crude.head())
print(crude.tail())

Crude shape: (10195, 3)
   timestamp  value released_at
0 1986-01-02  25.56  1986-01-02
1 1986-01-03  26.00  1986-01-03
2 1986-01-06  26.53  1986-01-06
3 1986-01-07  25.85  1986-01-07
4 1986-01-08  25.87  1986-01-08
       timestamp  value released_at
10190 2026-06-29  71.87  2026-06-29
10191 2026-06-30  70.56  2026-06-30
10192 2026-07-01  69.74  2026-07-01
10193 2026-07-02  69.73  2026-07-02
10194 2026-07-06  69.60  2026-07-06


In [10]:
# Convert daily crude to monthly (take the month's average)
crude_monthly = crude.copy()
crude_monthly['month'] = crude_monthly['timestamp'].dt.to_period('M').dt.to_timestamp()
crude_monthly = crude_monthly.groupby('month')['value'].mean().reset_index()
crude_monthly.columns = ['timestamp', 'crude']

print(f"Crude monthly shape: {crude_monthly.shape}")
print(crude_monthly.head())

# Merge crude into our model data
merged2 = pd.merge(
    merged[['timestamp', 'gasoline', 'indpro']],
    crude_monthly,
    on='timestamp',
    how='inner'
)

# Compute percent changes
merged2 = merged2.sort_values('timestamp').reset_index(drop=True)
merged2['gasoline_pct'] = merged2['gasoline'].pct_change()
merged2['indpro_pct'] = merged2['indpro'].pct_change()
merged2['crude_pct'] = merged2['crude'].pct_change()

model_data2 = merged2.dropna().reset_index(drop=True)

print(f"\nModel data shape: {model_data2.shape}")
print(f"Date range: {model_data2['timestamp'].min()} to {model_data2['timestamp'].max()}")
print("\nCorrelations with gasoline change:")
print(model_data2[['gasoline_pct', 'indpro_pct', 'crude_pct']].corr()['gasoline_pct'])

Crude monthly shape: (487, 2)
   timestamp      crude
0 1986-01-01  22.925455
1 1986-02-01  15.454737
2 1986-03-01  12.612500
3 1986-04-01  12.843636
4 1986-05-01  15.377619

Model data shape: (484, 7)
Date range: 1986-02-01 00:00:00 to 2026-05-01 00:00:00

Correlations with gasoline change:
gasoline_pct    1.000000
indpro_pct      0.217360
crude_pct       0.613214
Name: gasoline_pct, dtype: float64


In [11]:
from sklearn.linear_model import LinearRegression

# Features (X) and target (y)
X = model_data2[['indpro_pct', 'crude_pct']].values
y = model_data2['gasoline_pct'].values

# Fit
reg = LinearRegression()
reg.fit(X, y)

print("Regression: gasoline_change ~ indpro_change + crude_change")
print(f"  Intercept:        {reg.intercept_:.5f}")
print(f"  Indpro coef:      {reg.coef_[0]:.5f}")
print(f"  Crude coef:       {reg.coef_[1]:.5f}")
print(f"  R-squared:        {reg.score(X, y):.4f}")

# Residuals (for uncertainty later)
predictions = reg.predict(X)
residuals = y - predictions
print(f"  Residual std:     {residuals.std():.5f}")

Regression: gasoline_change ~ indpro_change + crude_change
  Intercept:        0.00122
  Indpro coef:      0.19269
  Crude coef:       0.29194
  R-squared:        0.3774
  Residual std:     0.03718


In [14]:
import inspect
from aieng.forecasting.methods.baselines.naive import LastValuePredictor

source = inspect.getsource(LastValuePredictor)
for line in source.splitlines():
    print(line)

class LastValuePredictor(Predictor):
    """Naive baseline: forecast the most recently observed value at all quantiles.

    All quantile levels receive the same value as the point forecast, producing
    a degenerate distribution with zero spread. This gives the worst possible
    calibration score — a well-calibrated model should spread its quantiles to
    reflect genuine uncertainty.

    For multi-horizon tasks (``len(task.horizons) > 1``), the same last value
    is carried forward as a flat forecast for every requested step — equivalent
    to the "persistence" or "random-walk" assumption.

    Parameters
    ----------
    None
    """

    # ------------------------------------------------------------------
    # Step 1: give your predictor a stable string ID.
    # This appears in BacktestResult and every Prediction record,
    # so changing it mid-experiment will break comparisons.
    # ------------------------------------------------------------------
    @property
    def

In [ ]:
import inspect
from aieng.forecasting.methods.baselines.naive import LastValuePredictor
print(inspect.getsource(LastValuePredictor))

class LastValuePredictor(Predictor):
    """Naive baseline: forecast the most recently observed value at all quantiles.

    All quantile levels receive the same value as the point forecast, producing
    a degenerate distribution with zero spread. This gives the worst possible
    calibration score — a well-calibrated model should spread its quantiles to
    reflect genuine uncertainty.

    For multi-horizon tasks (``len(task.horizons) > 1``), the same last value
    is carried forward as a flat forecast for every requested step — equivalent
    to the "persistence" or "random-walk" assumption.

    Parameters
    ----------
    None
    """

    # ------------------------------------------------------------------
    # Step 1: give your predictor a stable string ID.
    # This appears in BacktestResult and every Prediction record,
    # so changing it mid-experiment will break comparisons.
    # ------------------------------------------------------------------
    @property
    def

In [16]:
# Find where ContinuousForecast, STANDARD_QUANTILES, Prediction, etc. actually live.
# The naive.py file imports them, so let's read its import lines.
import inspect
from aieng.forecasting.methods.baselines import naive

src = inspect.getsource(naive)
for line in src.splitlines():
    if line.startswith("from ") or line.startswith("import "):
        print(line)

from __future__ import annotations
from datetime import datetime, timezone
import pandas as pd
from aieng.forecasting.data.context import ForecastContext
from aieng.forecasting.evaluation.prediction import STANDARD_QUANTILES, ContinuousForecast, Prediction
from aieng.forecasting.evaluation.predictor import Predictor
from aieng.forecasting.evaluation.task import ForecastingTask


In [17]:
from aieng.forecasting.data.context import ForecastContext
from aieng.forecasting.evaluation.prediction import STANDARD_QUANTILES, ContinuousForecast, Prediction
from aieng.forecasting.evaluation.predictor import Predictor
from aieng.forecasting.evaluation.task import ForecastingTask
print("STANDARD_QUANTILES:", STANDARD_QUANTILES)
print("imports OK")

STANDARD_QUANTILES: [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
imports OK


In [18]:
from __future__ import annotations
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from scipy.stats import norm

from aieng.forecasting.data.context import ForecastContext
from aieng.forecasting.evaluation.prediction import STANDARD_QUANTILES, ContinuousForecast, Prediction
from aieng.forecasting.evaluation.predictor import Predictor
from aieng.forecasting.evaluation.task import ForecastingTask


class MacroRegressionPredictor(Predictor):
    """Forecast gasoline % change from LAGGED macro drivers (crude, indpro).

    Model (refit at every origin on data available at as_of):
        gasoline_pct_t  ~  crude_pct_{t-1}  +  indpro_pct_{t-1}

    Lagged predictors mean every input is already known when the forecast
    is issued — no look-ahead. Uncertainty comes from a Gaussian spread of
    the in-sample residuals.
    """

    def __init__(self, crude_id="crude", indpro_id="indpro"):
        self.crude_id = crude_id
        self.indpro_id = indpro_id

    @property
    def predictor_id(self) -> str:
        return "macro_regression_lag1"

    def _monthly(self, df):
        """Collapse any series to month-start frequency (mean within month)."""
        out = df.copy()
        out["timestamp"] = out["timestamp"].dt.to_period("M").dt.to_timestamp()
        return out.groupby("timestamp")["value"].mean().reset_index()

    def predict(self, task: ForecastingTask, context: ForecastContext) -> list[Prediction]:
        # fetch everything (already cut to as_of, no leakage)
        target = self._monthly(context.get_series(task.target_series_id)).rename(columns={"value": "gas"})
        crude = self._monthly(context.get_series(self.crude_id)).rename(columns={"value": "crude"})
        indpro = self._monthly(context.get_series(self.indpro_id)).rename(columns={"value": "indpro"})

        df = (target.merge(crude, on="timestamp", how="inner")
                    .merge(indpro, on="timestamp", how="inner")
                    .sort_values("timestamp").reset_index(drop=True))

        df["gas_pct"] = df["gas"].pct_change()
        df["crude_pct"] = df["crude"].pct_change()
        df["indpro_pct"] = df["indpro"].pct_change()

        # lag predictors by 1 month
        df["crude_pct_lag1"] = df["crude_pct"].shift(1)
        df["indpro_pct_lag1"] = df["indpro_pct"].shift(1)

        fit_df = df.dropna(subset=["gas_pct", "crude_pct_lag1", "indpro_pct_lag1"])

        last_value = float(target["gas"].iloc[-1])

        if len(fit_df) < 24:
            # not enough history — fall back to persistence
            payload = ContinuousForecast(
                point_forecast=last_value,
                quantiles=dict.fromkeys(STANDARD_QUANTILES, last_value),
            )
        else:
            X = fit_df[["crude_pct_lag1", "indpro_pct_lag1"]].values
            y = fit_df["gas_pct"].values
            reg = LinearRegression().fit(X, y)
            resid_std = float((y - reg.predict(X)).std())

            # next-step predictors = most recent KNOWN monthly changes
            x_next = df[["crude_pct", "indpro_pct"]].iloc[-1].values.reshape(1, -1)
            pred_pct = float(reg.predict(x_next)[0])

            point = last_value * (1 + pred_pct)
            quantiles = {
                q: last_value * (1 + pred_pct + norm.ppf(q) * resid_std)
                for q in STANDARD_QUANTILES
            }
            payload = ContinuousForecast(point_forecast=point, quantiles=quantiles)

        offset = pd.tseries.frequencies.to_offset(task.frequency)
        issued_at = datetime.now(tz=timezone.utc).replace(tzinfo=None)

        return [
            Prediction(
                predictor_id=self.predictor_id,
                task_id=task.task_id,
                issued_at=issued_at,
                as_of=context.as_of,
                forecast_date=(pd.Timestamp(context.as_of) + offset * h).to_pydatetime(),
                payload=payload,
            )
            for h in task.horizons
        ]

print("MacroRegressionPredictor defined")

MacroRegressionPredictor defined


In [19]:
# Confirm all needed series are registered
print("Registered series:")
print(svc.summary()[["series_id"]])

Registered series:
             series_id
0  cpi_gasoline_canada
1                crude
2               indpro


In [21]:
import yaml
from pathlib import Path
from aieng.forecasting.evaluation import BacktestSpec, backtest
from aieng.forecasting.methods.baselines.naive import LastValuePredictor
from aieng.forecasting.methods.numerical.darts_arima import DartsAutoARIMAPredictor

# Load the reference spec
spec_path = ROOT / "implementations" / "getting_started" / "specs" / "cpi_gasoline_1m.yaml"
with spec_path.open() as f:
    spec = BacktestSpec.model_validate(yaml.safe_load(f))

# Baselines
naive_results = backtest(predictor=LastValuePredictor(), spec=spec, data_service=svc)
arima_results = backtest(predictor=DartsAutoARIMAPredictor(num_samples=500), spec=spec, data_service=svc)

print("baselines done")

baselines done


In [22]:
from aieng.forecasting.evaluation import backtest

macro_predictor = MacroRegressionPredictor(crude_id="crude", indpro_id="indpro")
macro_results = backtest(predictor=macro_predictor, spec=spec, data_service=svc)

print(f"{'Predictor':<28} {'Origins':>8} {'Skipped':>8} {'Mean CRPS':>10}")
print("-" * 58)
print(f"{'naive':<28} {len(naive_results.predictions):>8} {naive_results.skipped_origins:>8} {naive_results.mean_score:>10.4f}")
print(f"{'AutoARIMA':<28} {len(arima_results.predictions):>8} {arima_results.skipped_origins:>8} {arima_results.mean_score:>10.4f}")
print(f"{'macro_regression_lag1':<28} {len(macro_results.predictions):>8} {macro_results.skipped_origins:>8} {macro_results.mean_score:>10.4f}")

Predictor                     Origins  Skipped  Mean CRPS
----------------------------------------------------------
naive                             301        0    10.1083
AutoARIMA                         301        0     8.4611
macro_regression_lag1             301        0     7.6903


In [23]:
def scores_by_date(result):
    return pd.DataFrame({
        "forecast_date": [p.forecast_date.date() for p in result.predictions],
        "crps": result.scores,
    }).set_index("forecast_date")

arima_s = scores_by_date(arima_results).rename(columns={"crps": "crps_arima"})
macro_s = scores_by_date(macro_results).rename(columns={"crps": "crps_macro"})

cmp = arima_s.join(macro_s)
cmp["improvement"] = cmp["crps_arima"] - cmp["crps_macro"]

# Where macro helped most
print("Biggest improvements over AutoARIMA:")
print(cmp.sort_values("improvement", ascending=False).head(8))
print("\nWhere macro did WORSE:")
print(cmp.sort_values("improvement").head(5))

Biggest improvements over AutoARIMA:
               crps_arima  crps_macro  improvement
forecast_date                                     
2022-12-01      48.979680   37.260618    11.719061
2008-11-01      57.902069   48.300021     9.602049
2022-03-01      36.693990   28.242722     8.451268
2022-06-01      52.803035   44.398899     8.404137
2023-02-01      13.581796    5.464928     8.116868
2006-07-01      10.604305    3.436430     7.167875
2021-11-01      10.130876    3.174958     6.955917
2017-04-01      15.421181    8.604130     6.817051

Where macro did WORSE:
               crps_arima  crps_macro  improvement
forecast_date                                     
2009-02-01       3.531879   18.889336   -15.357457
2020-06-01      30.951526   44.209202   -13.257675
2020-05-01       2.212001    7.948471    -5.736470
2005-12-01       8.992778   14.697577    -5.704799
2023-11-01      17.402899   23.053941    -5.651042
